In [1]:
from os import environ

import pandas as pd
import numpy as np
import os
PATH_SUPPLEMENTARY = os.path.join(os.getcwd(), "41588_2024_1854_MOESM3_ESM.xlsx")

df_pred_bert = pd.read_excel(PATH_SUPPLEMENTARY, sheet_name="ST5", skiprows=2)

In [2]:
from datasets import load_dataset

ds = load_dataset("opentargets/clinical_trial_reason_to_stop")
df_train = ds["train"].to_pandas()
assert "test" not in ds

In [3]:
print(df_train.text.nunique())
df_train

3747


,text,label_descriptions,Another_Study,Business_Administrative,Covid19,Endpoint_Met,Ethical_Reason,Insufficient_Data,Insufficient_Enrollment,Interim_Analysis,Invalid_Reason,Logistics_Resources,Negative,No_Context,Regulatory,Safety_Sideeffects,Study_Design,Study_Staff_Moved,Success
0,&quot;Tapering doses&quot; protocol arm was no...,[Negative],False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,1 Indication for further investigations (brain...,[Study_Design],False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,1. Very low enrollment rate. Ê Ê 2. Recent stu...,"[Insufficient_Enrollment, Safety_Sideeffects, ...",True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False
3,10/2005 PI assigned duties as trauma physician...,[Business_Administrative],False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,12/15/2008 Voluntarily placed on inactive stat...,[Invalid_Reason],False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742,we decided not to go on treatment phase,[Invalid_Reason],False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3743,we didnt recieved the medicine,[Logistics_Resources],False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
3744,"we have seen little, if any, improved benefit ...",[Negative],False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3745,withdrawn due to contractual issues,[Business_Administrative],False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [4]:
df_train.Negative.value_counts() / df_train.shape[0]

Negative
False    0.901788
True     0.098212
Name: count, dtype: float64

In [5]:
# Manual analysis of the co-labels to identify the records to exclude.
df_train[~df_train.label_descriptions.apply(lambda x: list(x) == ["Negative"]) & (df_train.Negative)].sort_values(
    by='label_descriptions', key=lambda x: x.astype(str))

,text,label_descriptions,Another_Study,Business_Administrative,Covid19,Endpoint_Met,Ethical_Reason,Insufficient_Data,Insufficient_Enrollment,Interim_Analysis,Invalid_Reason,Logistics_Resources,Negative,No_Context,Regulatory,Safety_Sideeffects,Study_Design,Study_Staff_Moved,Success
1807,Publication of recent trials showing no effect...,"[Another_Study, Negative]",True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1149,Lack of efficacy demonstrated in study ICA-170...,"[Another_Study, Negative]",True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
2422,Study terminated on 15DEC2016 due to study A82...,"[Another_Study, Negative]",True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1171,Lack of improved efficacy compared to historic...,"[Another_Study, Negative]",True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1280,"Lucentis, a treatment superior to PDT, has bec...","[Another_Study, Negative]",True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2823,The results of the dose escalation phase did n...,"[Study_Design, Safety_Sideeffects, Negative]",False,False,False,False,False,False,False,False,False,False,True,False,False,True,True,False,False
2952,The study was terminated due to hepatoxicity o...,"[Study_Design, Safety_Sideeffects, Negative]",False,False,False,False,False,False,False,False,False,False,True,False,False,True,True,False,False
1544,Original investigator left the NIH and the pri...,"[Study_Staff_Moved, Negative]",False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
2936,The study was terminated as the Prinicipal Inv...,"[Study_Staff_Moved, Negative]",False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False


### New dataset parsing / generation

In [6]:
# After a manual review, the following co-labels are to be thrown away.
colabels_to_exclude = [
    "Business_Administrative",
    "Ethical_Reason",
    "Insufficient_Enrollment",
    "Regulatory",
    "Safety_Sideeffects",
    "Study_Staff_Moved",
    "Success"
]
def is_negative(label_descriptions):
    ld = str(label_descriptions)
    return "Negative" in ld and not any(cte in ld for cte in colabels_to_exclude)
df_train_neg = df_train[["text", "label_descriptions"]].copy()
df_train_neg["negative_efficacy"] = df_train.label_descriptions.apply(is_negative)
df_train_neg

,text,label_descriptions,negative_efficacy
0,&quot;Tapering doses&quot; protocol arm was no...,[Negative],True
1,1 Indication for further investigations (brain...,[Study_Design],False
2,1. Very low enrollment rate. Ê Ê 2. Recent stu...,"[Insufficient_Enrollment, Safety_Sideeffects, ...",False
3,10/2005 PI assigned duties as trauma physician...,[Business_Administrative],False
4,12/15/2008 Voluntarily placed on inactive stat...,[Invalid_Reason],False
...,...,...,...
3742,we decided not to go on treatment phase,[Invalid_Reason],False
3743,we didnt recieved the medicine,[Logistics_Resources],False
3744,"we have seen little, if any, improved benefit ...",[Negative],True
3745,withdrawn due to contractual issues,[Business_Administrative],False


In [7]:
# Add embeddings - tf-IDF and fasttext
import fasttext

# Run at least once per installation.
# import fasttext.util
# fasttext.util.download_model('en', if_exists='ignore')

ft = fasttext.load_model("cc.en.300.bin")

# Compute mean vector of word embeddings
def embed_fasttext(text):
    toks = text.lower().split()
    if not toks:
        return np.zeros(ft.get_dimension())
    return np.mean([ft.get_word_vector(t) for t in toks], axis=0)

df_train_neg["vec_fasttext"] = df_train_neg.text.apply(embed_fasttext)

In [59]:
import twinning

twin_indices = twinning.twin(np.array(df_train_neg.vec_fasttext.tolist()), r=7,u1=42)
df_train_neg['split'] = 'train'
df_train_neg.iloc[twin_indices, df_train_neg.columns.get_loc('split')] = 'test'
df_train_neg[["split", "negative_efficacy"]].value_counts()


split  negative_efficacy
train  False                2928
test   False                 489
train  True                  283
test   True                   47
Name: count, dtype: int64

In [78]:
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve, balanced_accuracy_score
)

def evaluate_binary(y_true, y_prob, threshold=0.5, title=""):
    """
    y_true: array of 0/1
    y_prob: array of P(y=1)
    threshold: decision boundary for predicted class
    """
    y_prob = np.array(y_prob)
    y_true = np.array(y_true)

    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, digits=3)
    auroc  = roc_auc_score(y_true, y_prob)
    auprc  = average_precision_score(y_true, y_prob)
    cm     = confusion_matrix(y_true, y_pred)
    ba     = balanced_accuracy_score(y_true, y_pred)

    print(f"=== {title or 'Evaluation'} ===")
    print(report)
    print("Confusion matrix:\n", cm)
    print(f"AUROC: {auroc:.3f} | AUPRC (pos class): {auprc:.3f} | B.Acc: {ba:.3f}")

    # Return a dict so you can log/compare later (including LLM runs)
    return {
        "report": report,
        "confusion_matrix": cm,
        "auroc": auroc,
        "auprc": auprc,
        "balanced_accuracy": ba,
        "threshold": threshold,
    }

def split_xy(df):
    """Strict split: expects df columns: text, negative_efficacy, vec_fasttext, split."""
    train = df[df["split"] == "train"].copy()
    test  = df[df["split"] == "test"].copy()
    Xtr_text, Xte_text = train["text"].tolist(), test["text"].tolist()
    ytr = train["negative_efficacy"].astype(int).to_numpy()
    yte = test["negative_efficacy"].astype(int).to_numpy()
    Xtr_vec = np.vstack(train["vec_fasttext"].to_numpy())
    Xte_vec = np.vstack(test["vec_fasttext"].to_numpy())
    return (Xtr_text, ytr, Xtr_vec), (Xte_text, yte, Xte_vec)

In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# Split
(Xtr_text, ytr, _Xtr_vec), (Xte_text, yte, _Xte_vec) = split_xy(df_train_neg)

# Pipeline: TF-IDF + calibrated LR
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", CalibratedClassifierCV(
        estimator=LogisticRegression(
            max_iter=1000, class_weight="balanced", solver="liblinear"
        ),
        method="isotonic", cv=3
    ))
])

tfidf_lr.fit(Xtr_text, ytr)
proba_tfidf = tfidf_lr.predict_proba(Xte_text)[:, 1]
res_tfidf = evaluate_binary(yte, proba_tfidf, title="TF-IDF → Calibrated Logistic Regression")


=== TF-IDF → Calibrated Logistic Regression ===
              precision    recall  f1-score   support

           0      0.973     0.969     0.971       489
           1      0.694     0.723     0.708        47

    accuracy                          0.948       536
   macro avg      0.834     0.846     0.840       536
weighted avg      0.949     0.948     0.948       536

Confusion matrix:
 [[474  15]
 [ 13  34]]
AUROC: 0.963 | AUPRC (pos class): 0.736 | B.Acc: 0.846


In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# Split
(Xtr_text, ytr, _Xtr_vec), (Xte_text, yte, _Xte_vec) = split_xy(df_train_neg)

# Pipeline: TF-IDF + Random Forest
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", RandomForestClassifier(random_state=42, n_estimators=100)
    )
])

tfidf_lr.fit(Xtr_text, ytr)
proba_tfidf = tfidf_lr.predict_proba(Xte_text)[:, 1]
res_tfidf = evaluate_binary(yte, proba_tfidf, title="TF-IDF → Random Forest")


=== TF-IDF → Random Forest ===
              precision    recall  f1-score   support

           0      0.954     0.980     0.967       489
           1      0.706     0.511     0.593        47

    accuracy                          0.938       536
   macro avg      0.830     0.745     0.780       536
weighted avg      0.932     0.938     0.934       536

Confusion matrix:
 [[479  10]
 [ 23  24]]
AUROC: 0.954 | AUPRC (pos class): 0.688 | B.Acc: 0.745


In [81]:
from sklearn.linear_model import LogisticRegression

# Split (uses vec_fasttext from df)
(_Xtr_text, ytr, Xtr_vec), (_Xte_text, yte, Xte_vec) = split_xy(df_train_neg)

ft_lr = LogisticRegression(max_iter=1000, class_weight="balanced")
ft_lr.fit(Xtr_vec, ytr)
proba_ft = ft_lr.predict_proba(Xte_vec)[:, 1]
res_ft = evaluate_binary(yte, proba_ft, title="fastText (mean) → Logistic Regression")

=== fastText (mean) → Logistic Regression ===
              precision    recall  f1-score   support

           0      0.980     0.818     0.892       489
           1      0.305     0.830     0.446        47

    accuracy                          0.819       536
   macro avg      0.643     0.824     0.669       536
weighted avg      0.921     0.819     0.853       536

Confusion matrix:
 [[400  89]
 [  8  39]]
AUROC: 0.892 | AUPRC (pos class): 0.469 | B.Acc: 0.824


In [82]:
from sklearn.ensemble import RandomForestClassifier

# Split (uses vec_fasttext from df)
(_Xtr_text, ytr, Xtr_vec), (_Xte_text, yte, Xte_vec) = split_xy(df_train_neg)

ft_lr = RandomForestClassifier(random_state=42, n_estimators=100)
ft_lr.fit(Xtr_vec, ytr)
proba_ft = ft_lr.predict_proba(Xte_vec)[:, 1]
res_ft = evaluate_binary(yte, proba_ft, title="fastText (mean) → Random Forest")

=== fastText (mean) → Random Forest ===
              precision    recall  f1-score   support

           0      0.917     0.998     0.956       489
           1      0.750     0.064     0.118        47

    accuracy                          0.916       536
   macro avg      0.834     0.531     0.537       536
weighted avg      0.903     0.916     0.882       536

Confusion matrix:
 [[488   1]
 [ 44   3]]
AUROC: 0.844 | AUPRC (pos class): 0.436 | B.Acc: 0.531


In [67]:
df_train[df_train.label_descriptions.apply(lambda x: "Negative" in str(x) and not any(cte in str(x) for cte in colabels_to_exclude))].label_descriptions.apply(str).unique()

array(["['Negative']", "['Study_Design' 'Negative']",
       "['Logistics_Resources' 'Study_Design' 'Negative']",
       "['Another_Study' 'Negative']", "['Endpoint_Met' 'Negative']",
       "['No_Context' 'Negative']"], dtype=object)

### LLM-based retrieval

In [89]:
import hashlib
import shelve
from contextlib import contextmanager
from pydantic import BaseModel, Field, field_validator

from openai import OpenAI
from instructor import from_openai

client = from_openai(OpenAI())

class LabelResponse(BaseModel):
    """Structured classification output for efficacy-failure detection."""

    label: int = Field(..., description="1 if lack of efficacy/futility/endpoints-not-met; else 0")

    confidence: float = Field(..., ge=0.0, le=1.0, description="Model certainty in [0,1]")

    reason: str = Field(..., min_length=1, max_length=300, description="<= 50 words concise justification")

    @field_validator("confidence")
    @classmethod
    def conf(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("confidence must be in [0,1]")
        return v

    @field_validator("label")
    @classmethod
    def _lbl01(cls, v):
        if v not in (0, 1):
            raise ValueError("label must be 0 or 1")
        return v

    @property
    def proba(self):
        # Map (label, confidence) -> probability of positive class (efficacy=1)
        conf = float(self.confidence)
        return conf if self.label == 1 else (1.0 - conf)

def _cache_key(model: str, system: str, text: str) -> str:
    h = hashlib.sha256()
    h.update(("MODEL:"+model+"\n").encode())
    h.update(("SYSTEM:"+system+"\n").encode())
    h.update(("TEXT:"+text).encode())
    return h.hexdigest()

@contextmanager
def open_cache(path):
    db = shelve.open(path)  # gdbm/ndbm under the hood
    try:
        yield db
    finally:
        db.close()


from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from typing import Callable


def _process_single_text(
    text: str,
    prediction_func: Callable[[str], LabelResponse],
    cache_path: str,
    model: str,
    system: str,
    overwrite: bool = False
) -> LabelResponse:
    """Process a single text with caching."""
    with open_cache(cache_path) as db:
        key = _cache_key(model, system, text)
        if (not overwrite) and (key in db):
            # Rehydrate Pydantic model from stored dict
            data = db[key]
            try:
                return LabelResponse(**data)
            except Exception:
                pass

        # Not cached or overwrite -> call prediction function
        obj = prediction_func(text)
        # Store as plain dict (robust to Pydantic changes)
        db[key] = obj.model_dump()
        return obj


def predict_batch_parallel(
    texts: list[str],
    prediction_func: Callable[[str], LabelResponse],
    cache_path: str,
    batch_size: int = 10,
    max_workers: int = None,
    model: str = None,
    system: str = None,
    overwrite: bool = False
) -> list[LabelResponse]:
    """
    Parallelize batch prediction with configurable parameters.

    Args:
        texts: List of texts to process
        prediction_func: Function that takes text and returns LabelResponse
        batch_size: Number of texts to process in parallel batches
        max_workers: Maximum number of threads (defaults to min(batch_size, 32))
        cache_path: Path to cache database
        model: Model identifier for caching
        system: System prompt for caching
        overwrite: Whether to overwrite cached results

    Returns:
        List of LabelResponse objects in the same order as input texts
    """
    # Set defaults
    if max_workers is None:
        max_workers = min(batch_size, 32)

    results = [None] * len(texts)

    # Process in batches
    with tqdm(total=len(texts), desc="Processing texts") as pbar:
        for batch_start in range(0, len(texts), batch_size):
            batch_end = min(batch_start + batch_size, len(texts))
            batch_texts = texts[batch_start:batch_end]
            batch_indices = list(range(batch_start, batch_end))

            # Process current batch in parallel
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                # Submit all tasks in current batch
                future_to_index = {
                    executor.submit(
                        _process_single_text,
                        text,
                        prediction_func,
                        cache_path,
                        model,
                        system,
                        overwrite
                    ): idx
                    for idx, text in zip(batch_indices, batch_texts)
                }

                # Collect results as they complete
                for future in as_completed(future_to_index):
                    idx = future_to_index[future]
                    try:
                        result = future.result()
                        results[idx] = result
                        pbar.update(1)
                    except Exception as exc:
                        print(f'Text at index {idx} generated an exception: {exc}')
                        # You might want to create a default response or re-raise
                        raise exc

    return results


In [90]:
# S0 - Zero shot
MODEL_S0 = "gpt-5-nano"
SYSTEM_S0 = (
    "You classify clinical-trial stoppage reasons.\n"
    "Return: label (0/1), confidence (0-1), reason (<=50 words). "
    "Label=1 if text implies lack of efficacy (futility, primary endpoints not met, no clinical benefit). "
    "Otherwise 0 (safety/adverse events/toxicity; recruitment/logistics; funding/business; regulatory/external/unknown). "
    "If unclear, output 0."
)


def _s0_prediction_func(text: str) -> LabelResponse:
    return client.chat.completions.create(
        model=MODEL_S0,
        response_model=LabelResponse,
        messages=[
            {"role": "system", "content": SYSTEM_S0},
            {"role": "user", "content": f'Text: "{text}"'}
        ],
    )

s0_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s0_prediction_func, cache_path="cache_s0", model=MODEL_S0, system=SYSTEM_S0)
proba_s0 = [o.proba for o in s0_struct]
res_s0 = evaluate_binary(yte, proba_s0, title="S0 - Zero-shot")


Processing texts: 100%|██████████| 536/536 [00:00<00:00, 2747.00it/s]


=== S0 - Zero-shot ===
              precision    recall  f1-score   support

           0      0.973     0.943     0.957       489
           1      0.548     0.723     0.624        47

    accuracy                          0.924       536
   macro avg      0.760     0.833     0.791       536
weighted avg      0.935     0.924     0.928       536

Confusion matrix:
 [[461  28]
 [ 13  34]]
AUROC: 0.919 | AUPRC (pos class): 0.655 | B.Acc: 0.833


In [144]:
print(SYSTEM_S0)

You classify clinical-trial stoppage reasons.
Return: label (0/1), confidence (0-1), reason (<=50 words). Label=1 if text implies lack of efficacy (futility, primary endpoints not met, no clinical benefit). Otherwise 0 (safety/adverse events/toxicity; recruitment/logistics; funding/business; regulatory/external/unknown). If unclear, output 0.


In [92]:
# S1 - Zero shot with cue
MODEL_S1 = "gpt-5-nano"
SYSTEM_S1 = (
    "You classify clinical-trial stoppage reasons.\n"
    "Return: label (0/1), confidence (0-1), reason (<=50 words). "
    "Label=1 if text implies lack of efficacy (futility, primary endpoints not met, no clinical benefit). "
    "Otherwise 0 (safety/adverse events/toxicity; recruitment/logistics; funding/business; regulatory/external/unknown). "
    "If unclear, output 0."
)


def _s1_prediction_func(text: str) -> LabelResponse:
    return client.chat.completions.create(
        model=MODEL_S1,
        response_model=LabelResponse,
        messages=[
            {"role": "system", "content": SYSTEM_S1},
             {"role":"user","content":f'Schema: label in {{0,1}}, confidence in [0,1].\nText: "{text}"'}
        ],
    )

s1_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s1_prediction_func, cache_path="cache_s1", model=MODEL_S1, system=SYSTEM_S1)
proba_s1 = [o.proba for o in s1_struct]
res_s1 = evaluate_binary(yte, proba_s1, title="S1 - Zero-shot + Schema Cue")


Processing texts: 100%|██████████| 536/536 [12:34<00:00,  1.41s/it]  

=== S1 - Zero-shot + Schema Cue ===
              precision    recall  f1-score   support

           0      0.971     0.949     0.960       489
           1      0.569     0.702     0.629        47

    accuracy                          0.927       536
   macro avg      0.770     0.826     0.794       536
weighted avg      0.935     0.927     0.931       536

Confusion matrix:
 [[464  25]
 [ 14  33]]
AUROC: 0.924 | AUPRC (pos class): 0.617 | B.Acc: 0.826


In [121]:
# S2 - Few shots
import json

MODEL_S2 = "gpt-5-nano"
# format demonstrations

example_pairs = [
    ("A higher rate of late rejection was seen in the low tacrolimus arm.", True),
    ("A preliminary analysis of 11 patients confirmed the safety profile of NCX-1000 but did not demonstrate the efficacy required.", True),
    ("A decision was made to not move forward with the study. No participants were enrolled or Ê treated.", False), # Business related
    ("A re-evaluation of research risks to participants were greater than originally anticipated", False) # Safety
]

_lines = []
for t,lbl in example_pairs:
    ex = {"label": int(lbl),
          "confidence": 0.9 if lbl==1 else 0.95,
          "reason": "Futility/endpoint miss => 1; otherwise non-efficacy => 0."}
    _lines.append(f'Text: "{t}"\n-> {json.dumps(ex)}')
SAMPLES_S2 = "\n\n".join(_lines)
SYSTEM_S2 = (
    "You classify clinical-trial stoppage reasons into efficacy failure (1) vs other (0). "
    "Return: label (0/1), confidence (0-1), reason (<=50 words). Use the demonstrations to stay consistent."
)

def _s2_prediction_func(text: str) -> LabelResponse:
    return client.chat.completions.create(
        model=MODEL_S2,
        response_model=LabelResponse,
        messages=[
            {"role": "system", "content": SYSTEM_S2},
            {"role":"user","content":f"Demonstrations:\n{SAMPLES_S2}\n\nClassify:\nText: \"{text}\""}
            ,
        ],
    )

s2_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s2_prediction_func, cache_path="cache_s2", model=MODEL_S2, system=SYSTEM_S2)
proba_s2 = [o.proba for o in s2_struct]
res_s2 = evaluate_binary(yte, proba_s2, title="S2 - Few-shot")


Processing texts: 100%|██████████| 536/536 [00:00<00:00, 1844.59it/s]

=== S2 - Few-shot ===
              precision    recall  f1-score   support

           0      0.979     0.953     0.966       489
           1      0.617     0.787     0.692        47

    accuracy                          0.938       536
   macro avg      0.798     0.870     0.829       536
weighted avg      0.947     0.938     0.942       536

Confusion matrix:
 [[466  23]
 [ 10  37]]
AUROC: 0.932 | AUPRC (pos class): 0.604 | B.Acc: 0.870


In [146]:
print(SYSTEM_S2)
print(SAMPLES_S2)

You classify clinical-trial stoppage reasons into efficacy failure (1) vs other (0). Return: label (0/1), confidence (0-1), reason (<=50 words). Use the demonstrations to stay consistent.
Text: "A higher rate of late rejection was seen in the low tacrolimus arm."
-> {"label": 1, "confidence": 0.9, "reason": "Futility/endpoint miss => 1; otherwise non-efficacy => 0."}

Text: "A preliminary analysis of 11 patients confirmed the safety profile of NCX-1000 but did not demonstrate the efficacy required."
-> {"label": 1, "confidence": 0.9, "reason": "Futility/endpoint miss => 1; otherwise non-efficacy => 0."}

Text: "A decision was made to not move forward with the study. No participants were enrolled or Ê treated."
-> {"label": 0, "confidence": 0.95, "reason": "Futility/endpoint miss => 1; otherwise non-efficacy => 0."}

Text: "A re-evaluation of research risks to participants were greater than originally anticipated"
-> {"label": 0, "confidence": 0.95, "reason": "Futility/endpoint miss => 

In [127]:
# S2(B) - Few shots - more capable reasoning , 5 examples
MODEL_S2B = "gpt-5-mini"
# format demonstrations

example_pairs_extended = example_pairs + [
    ("this study was suspended for futility", True),
    ("the study was halted after the first interim analysis as there were significantly more patients who recovered at one month in the control arm", True),
    ("primary efficacy analysis at Week 24 did not demonstrate non-inferiority of raltegravir versus lopinavir (+) ritonavir", True),
    ("Accrual goal for interventional part not achievable", False), # Insufficient enrollment
    ("the study has recently ended, all information gathred as planed", False), # Invalid reason
    ("patel no longer at pittsburgh", False), #Study staff moved
]

# Other examples:
#  "opposite result"
#  "performance variability"

assert len(example_pairs_extended) == 10

_lines = []
for t,lbl in example_pairs_extended:
    ex = {"label": int(lbl),
          "confidence": 0.9 if lbl==1 else 0.95,
          "reason": "Futility/endpoint miss => 1; otherwise non-efficacy => 0."}
    _lines.append(f'Text: "{t}"\n-> {json.dumps(ex)}')
SAMPLES_S2B = "\n\n".join(_lines)

SYSTEM_S2B = (
    "You classify clinical-trial stoppage reasons into efficacy failure (1) vs other (0). "
    "Return: label (0/1), confidence (0-1), reason (<=50 words). Use the demonstrations to stay consistent."
)

def _s2b_prediction_func(text: str) -> LabelResponse:
    return client.chat.completions.create(
        model=MODEL_S2B,
        response_model=LabelResponse,
        messages=[
            {"role": "system", "content": SYSTEM_S2B},
            {"role":"user","content":f"Demonstrations:\n{SAMPLES_S2B}\n\nClassify:\nText: \"{text}\""}
            ,
        ],
    )

s2b_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s2b_prediction_func, cache_path="cache_s2b", model=MODEL_S2B, system=SYSTEM_S2B)
proba_s2b = [o.proba for o in s2b_struct]
res_s2b = evaluate_binary(yte, proba_s2b, title="S2 - Few-shot - increased context, 5 positive, 5 negative examples")


Processing texts: 100%|██████████| 536/536 [00:00<00:00, 2973.71it/s]

=== S2 - Few-shot - increased context, 5 positive, 5 negative examples ===
              precision    recall  f1-score   support

           0      0.977     0.955     0.966       489
           1      0.621     0.766     0.686        47

    accuracy                          0.938       536
   macro avg      0.799     0.860     0.826       536
weighted avg      0.946     0.938     0.941       536

Confusion matrix:
 [[467  22]
 [ 11  36]]
AUROC: 0.950 | AUPRC (pos class): 0.650 | B.Acc: 0.860


In [129]:
# S3 Few-shots with rationale
MODEL_S3 = "gpt-5-mini"
_lines = []
for t,lbl in example_pairs_extended:
    ex = {"label": int(lbl),
          "confidence": 0.9 if lbl==1 else 0.95,
          "reason": "Concise justification reflecting label."}
    _lines.append(f'Text: "{t}"\n-> {json.dumps(ex)}')
SAMPLES_S3 = "\n\n".join(_lines)

SYSTEM_S3 = (
    "You classify clinical-trial stoppage reasons. Think briefly (<=2 sentences), "
    "then output JSON: label (0/1), confidence (0-1), reason (<=50 words). "
    "Label=1 for lack of efficacy/futility/primary endpoints not met; else 0."
)

def _s3_prediction_func(text: str) -> LabelResponse:
    return client.chat.completions.create(
        model=MODEL_S3,
        response_model=LabelResponse,
        messages=[
            {"role": "system", "content": SYSTEM_S3},
            {"role":"user","content":f"Demonstrations:\n{SAMPLES_S3}\n\nClassify with brief justification:\nText: \"{text}\""}
            ,
        ],
    )

s3_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s3_prediction_func, cache_path="cache_s3", model=MODEL_S3, system=SYSTEM_S3)
proba_s3 = [o.proba for o in s3_struct]
res_s3 = evaluate_binary(yte, proba_s3, title="S3 - Few-shot with rationale")

Processing texts: 100%|██████████| 536/536 [19:21<00:00,  2.17s/it]

=== S3 - Few-shot with rationale ===
              precision    recall  f1-score   support

           0      0.981     0.943     0.961       489
           1      0.576     0.809     0.673        47

    accuracy                          0.931       536
   macro avg      0.778     0.876     0.817       536
weighted avg      0.945     0.931     0.936       536

Confusion matrix:
 [[461  28]
 [  9  38]]
AUROC: 0.950 | AUPRC (pos class): 0.596 | B.Acc: 0.876


In [149]:
SAMPLES_S3

'Text: "A higher rate of late rejection was seen in the low tacrolimus arm."\n-> {"label": 1, "confidence": 0.9, "reason": "Concise justification reflecting label."}\n\nText: "A preliminary analysis of 11 patients confirmed the safety profile of NCX-1000 but did not demonstrate the efficacy required."\n-> {"label": 1, "confidence": 0.9, "reason": "Concise justification reflecting label."}\n\nText: "A decision was made to not move forward with the study. No participants were enrolled or Ê treated."\n-> {"label": 0, "confidence": 0.95, "reason": "Concise justification reflecting label."}\n\nText: "A re-evaluation of research risks to participants were greater than originally anticipated"\n-> {"label": 0, "confidence": 0.95, "reason": "Concise justification reflecting label."}\n\nText: "this study was suspended for futility"\n-> {"label": 1, "confidence": 0.9, "reason": "Concise justification reflecting label."}\n\nText: "the study was halted after the first interim analysis as there were

In [132]:
# S4 Few-shots - consistency
MODEL_S4 = "gpt-5-mini"
SAMPLES_S4 = SAMPLES_S3
SYSTEM_S4 = SYSTEM_S3

K_VOTES = 3

def _s4_prediction_func(text: str) -> LabelResponse:
    outs = [_s3_prediction_func(text + i * " " )  for i in range(K_VOTES)]
    lbls = np.array([int(o.label) for o in outs])
    conf = np.array([float(o.confidence) for o in outs])

    maj = int(lbls.sum() > (K_VOTES/2))
    conf_towards = np.where(lbls==maj, conf, 1.0-conf)
    p = float(conf_towards.mean())

    # concise reason (first non-empty)
    reason = next((o.reason for o in outs if o.reason and o.label == maj), "Aggregated by self-consistency.")
    return LabelResponse(label=maj, confidence=p, reason=reason[:300])

s4_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s4_prediction_func, cache_path="cache_s4", model=MODEL_S4, system=SYSTEM_S4)
proba_s4 = [o.proba for o in s4_struct]
res_s4 = evaluate_binary(yte, proba_s4, title="S4 - Few-shot with rationale - 3 vote consistency")

Processing texts: 100%|██████████| 536/536 [43:58<00:00,  4.92s/it]  

=== S4 - Few-shot with rationale - 3 vote consistency ===
              precision    recall  f1-score   support

           0      0.979     0.949     0.964       489
           1      0.597     0.787     0.679        47

    accuracy                          0.935       536
   macro avg      0.788     0.868     0.821       536
weighted avg      0.945     0.935     0.939       536

Confusion matrix:
 [[464  25]
 [ 10  37]]
AUROC: 0.944 | AUPRC (pos class): 0.703 | B.Acc: 0.868


In [139]:
# S5 - RAG
MODEL_S5 = "gpt-5-mini"
TOP_K = 10

def l2norm(X: np.ndarray) -> np.ndarray:
    nrm = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    return X / nrm

S5_Xn = l2norm(np.asarray(Xtr_vec))

SYSTEM_S5 = (
    "You classify clinical-trial stoppage reasons. Use retrieved labeled examples as guidance, "
    "but decide strictly from the provided text. Return: label (0/1), confidence (0-1), reason (<=50 words)."
)

TEST_T2V_LOOKUP = {t: v for t, v in zip(Xte_text, Xte_vec)}

def _build_samples(vec_query: np.ndarray) -> str:
    q = vec_query / (np.linalg.norm(vec_query) + 1e-12)
    sims = S5_Xn @ q
    top = sims.argsort()[::-1][:TOP_K]
    lines = []
    for j in top:
        ex = {"label": int(ytr[j]),
              "confidence": 0.9 if ytr[j]==1 else 0.95,
              "reason": "Retrieved labeled example."}
        lines.append(f'Text: "{Xtr_text[j]}"\n-> {json.dumps(ex)}')
    return "\n\n".join(lines)


def _s5_prediction_func(text: str) -> LabelResponse:
    vec_query = TEST_T2V_LOOKUP[text.strip()]
    samples = _build_samples(vec_query)
    return client.chat.completions.create(
        model=MODEL_S5,
        response_model=LabelResponse,
        messages=[
            {"role":"system","content":SYSTEM_S5},
            {"role":"user","content":f"Retrieved demonstrations:\n{samples}\n\nClassify:\nText: \"{text}\""}
        ],
    )

s5_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s5_prediction_func, cache_path="cache_s5", model=MODEL_S5, system=SYSTEM_S5)
proba_s5 = [o.proba for o in s5_struct]
res_s5 = evaluate_binary(yte, proba_s5, title="S5 - RAG")

Processing texts: 100%|██████████| 536/536 [00:00<00:00, 2736.56it/s]


=== S5 - RAG ===
              precision    recall  f1-score   support

           0      0.981     0.951     0.966       489
           1      0.613     0.809     0.697        47

    accuracy                          0.938       536
   macro avg      0.797     0.880     0.831       536
weighted avg      0.949     0.938     0.942       536

Confusion matrix:
 [[465  24]
 [  9  38]]
AUROC: 0.928 | AUPRC (pos class): 0.653 | B.Acc: 0.880


In [150]:
SYSTEM_S5

'You classify clinical-trial stoppage reasons. Use retrieved labeled examples as guidance, but decide strictly from the provided text. Return: label (0/1), confidence (0-1), reason (<=50 words).'

In [140]:
# S6 RAG + Self-consistency
MODEL_S6 = "gpt-5-mini"
SYSTEM_S6 = SYSTEM_S5
K_VOTES = 3

def _s6_prediction_func(text: str) -> LabelResponse:
    outs = [_s5_prediction_func(text + i * " " ) for i in range(K_VOTES)]
    lbls = np.array([int(o.label) for o in outs])
    conf = np.array([float(o.confidence) for o in outs])

    maj = int(lbls.sum() > (K_VOTES/2))
    conf_towards = np.where(lbls==maj, conf, 1.0-conf)
    p = float(conf_towards.mean())

    reason = next((o.reason for o in outs if o.reason and o.label == maj), "Aggregated by self-consistency.")
    return LabelResponse(label=maj, confidence=p, reason=reason[:300])

s6_struct = predict_batch_parallel(texts=Xte_text, prediction_func=_s6_prediction_func, cache_path="cache_s6", model=MODEL_S6, system=SYSTEM_S6)
proba_s6 = [o.proba for o in s6_struct]
res_s6 = evaluate_binary(yte, proba_s6, title="S6 - Few-shot with rationale - 3 vote consistency")

Processing texts: 100%|██████████| 536/536 [28:22<00:00,  3.18s/it]  

=== S6 - Few-shot with rationale - 3 vote consistency ===
              precision    recall  f1-score   support

           0      0.981     0.953     0.967       489
           1      0.623     0.809     0.704        47

    accuracy                          0.940       536
   macro avg      0.802     0.881     0.835       536
weighted avg      0.950     0.940     0.944       536

Confusion matrix:
 [[466  23]
 [  9  38]]
AUROC: 0.942 | AUPRC (pos class): 0.671 | B.Acc: 0.881


In [12]:
df_pred_bert

,Stop Reason,Phase,NCT_ID,Disease_ID,Therapeutic_Area,Start_Date,Overall_Status,Last_Update_Posted_Date,Completion_Date,Another_Study,...,Interim_Analysis,Invalid_Reason,Logistics_Resources,Negative,No_Context,Regulatory,Safety_Sideeffects,Study_Design,Study_Staff_Moved,Success
0,COVID19,NaN,NCT05075902,NaN,NaN,2019-02-01,Suspended,2021-10-13,2022-03-01,0,...,0,0,0,0,0,0,0,0,0,0
1,Low enrollment at the site.,NaN,NCT05067322,NaN,NaN,2017-01-09,Terminated,2021-10-05,2021-09-23,0,...,0,0,0,0,0,0,0,0,0,0
2,Change in lab priority,NaN,NCT05066906,NaN,NaN,2021-05-31,Withdrawn,2021-10-04,2021-07-31,0,...,0,0,0,0,0,0,0,0,0,0
3,Medical device expired,NaN,NCT05053789,NaN,NaN,2021-05-12,Terminated,2021-09-22,2021-08-27,0,...,0,0,1,0,0,0,0,0,0,0
4,Recruitment of participants was interrupted by...,NaN,NCT05051800,NaN,NaN,2019-02-01,Terminated,2021-09-21,2021-06-30,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28882,"No funding, move to expanded access",Phase 2,NCT00004418,NaN,NaN,1998-04-30,Terminated,2020-10-23,2014-12-31,0,...,0,0,0,0,0,0,0,0,0,0
28883,Study was never funded.,Phase 2,NCT00000340,NaN,NaN,1996-06-30,Withdrawn,2017-01-12,NaT,0,...,0,0,0,0,0,0,0,0,0,0
28884,Unable to recruit adequate number of subjects,Phase 2,NCT00000309,EFO_0002610,nervous system disease,1994-08-31,Terminated,2015-07-24,2000-01-31,0,...,0,0,0,0,0,0,0,0,0,0
28885,Original P.I. left the institution,Phase 1,NCT00000305,EFO_0002610,nervous system disease,NaT,Terminated,2012-05-18,NaT,0,...,0,0,0,0,0,0,0,0,1,0


In [19]:
df_pred_bert

,Stop Reason,Phase,NCT_ID,Disease_ID,Therapeutic_Area,Start_Date,Overall_Status,Last_Update_Posted_Date,Completion_Date,Another_Study,...,Interim_Analysis,Invalid_Reason,Logistics_Resources,Negative,No_Context,Regulatory,Safety_Sideeffects,Study_Design,Study_Staff_Moved,Success
0,COVID19,NaN,NCT05075902,NaN,NaN,2019-02-01,Suspended,2021-10-13,2022-03-01,0,...,0,0,0,0,0,0,0,0,0,0
1,Low enrollment at the site.,NaN,NCT05067322,NaN,NaN,2017-01-09,Terminated,2021-10-05,2021-09-23,0,...,0,0,0,0,0,0,0,0,0,0
2,Change in lab priority,NaN,NCT05066906,NaN,NaN,2021-05-31,Withdrawn,2021-10-04,2021-07-31,0,...,0,0,0,0,0,0,0,0,0,0
3,Medical device expired,NaN,NCT05053789,NaN,NaN,2021-05-12,Terminated,2021-09-22,2021-08-27,0,...,0,0,1,0,0,0,0,0,0,0
4,Recruitment of participants was interrupted by...,NaN,NCT05051800,NaN,NaN,2019-02-01,Terminated,2021-09-21,2021-06-30,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28882,"No funding, move to expanded access",Phase 2,NCT00004418,NaN,NaN,1998-04-30,Terminated,2020-10-23,2014-12-31,0,...,0,0,0,0,0,0,0,0,0,0
28883,Study was never funded.,Phase 2,NCT00000340,NaN,NaN,1996-06-30,Withdrawn,2017-01-12,NaT,0,...,0,0,0,0,0,0,0,0,0,0
28884,Unable to recruit adequate number of subjects,Phase 2,NCT00000309,EFO_0002610,nervous system disease,1994-08-31,Terminated,2015-07-24,2000-01-31,0,...,0,0,0,0,0,0,0,0,0,0
28885,Original P.I. left the institution,Phase 1,NCT00000305,EFO_0002610,nervous system disease,NaT,Terminated,2012-05-18,NaT,0,...,0,0,0,0,0,0,0,0,1,0


In [10]:
df_train

,text,label_descriptions,Another_Study,Business_Administrative,Covid19,Endpoint_Met,Ethical_Reason,Insufficient_Data,Insufficient_Enrollment,Interim_Analysis,Invalid_Reason,Logistics_Resources,Negative,No_Context,Regulatory,Safety_Sideeffects,Study_Design,Study_Staff_Moved,Success
0,&quot;Tapering doses&quot; protocol arm was no...,[Negative],False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,1 Indication for further investigations (brain...,[Study_Design],False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,1. Very low enrollment rate. Ê Ê 2. Recent stu...,"[Insufficient_Enrollment, Safety_Sideeffects, ...",True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False
3,10/2005 PI assigned duties as trauma physician...,[Business_Administrative],False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,12/15/2008 Voluntarily placed on inactive stat...,[Invalid_Reason],False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742,we decided not to go on treatment phase,[Invalid_Reason],False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3743,we didnt recieved the medicine,[Logistics_Resources],False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
3744,"we have seen little, if any, improved benefit ...",[Negative],False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3745,withdrawn due to contractual issues,[Business_Administrative],False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [28]:
#df_train.merge(df_pred_bert, how="left", left_on="text", right_on="Stop Reason")
# Verified the text - label description is unique, nothing fishy here.

,text,label_descriptions,Another_Study_x,Business_Administrative_x,Covid19_x,Endpoint_Met_x,Ethical_Reason_x,Insufficient_Data_x,Insufficient_Enrollment_x,Interim_Analysis_x,...,Interim_Analysis_y,Invalid_Reason_y,Logistics_Resources_y,Negative_y,No_Context_y,Regulatory_y,Safety_Sideeffects_y,Study_Design_y,Study_Staff_Moved_y,Success_y
0,&quot;Tapering doses&quot; protocol arm was no...,[Negative],False,False,False,False,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 Indication for further investigations (brain...,[Study_Design],False,False,False,False,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1. Very low enrollment rate. Ê Ê 2. Recent stu...,"[Insufficient_Enrollment, Safety_Sideeffects, ...",True,False,False,False,False,False,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10/2005 PI assigned duties as trauma physician...,[Business_Administrative],False,True,False,False,False,False,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,12/15/2008 Voluntarily placed on inactive stat...,[Invalid_Reason],False,False,False,False,False,False,False,False,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9042,we decided not to go on treatment phase,[Invalid_Reason],False,False,False,False,False,False,False,False,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9043,we didnt recieved the medicine,[Logistics_Resources],False,False,False,False,False,False,False,False,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9044,"we have seen little, if any, improved benefit ...",[Negative],False,False,False,False,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9045,withdrawn due to contractual issues,[Business_Administrative],False,True,False,False,False,False,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
